# 🎾 OMNIS-COURT LLM + Jina Server (v7.4 STABLE)
## transformers + 4-bit Quantization (No vLLM)

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run All (Ctrl+F9)
3. Wait ~8-12 minutes (model download + quantization)
4. Copy both URLs from Cell 5
5. Paste into config/platforms.json
6. Close tab (anti-idle active)

In [ ]:
# ==========================================
# CELL 1: INSTALL DEPENDENCIES (NO vLLM)
# ==========================================
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Uninstall any vLLM leftovers (prevents conflicts)
!pip uninstall -y vllm flashinfer-python humming-kernels -q 2>/dev/null

# Core packages (use existing torch/transformers from Colab)
!pip install -q bitsandbytes accelerate trafilatura fastapi uvicorn nest-asyncio requests

# Install cloudflared binary
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Verify installations
import subprocess
cf = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'\n✅ cloudflared: {cf.stdout.strip()}')

all_ok = True
for pkg in ['torch', 'transformers', 'bitsandbytes', 'accelerate', 'trafilatura', 'fastapi', 'uvicorn']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'✅ {pkg}')
    except Exception as e:
        print(f'❌ {pkg}: {e}')
        all_ok = False

if all_ok:
    print('\n✅ ALL dependencies ready!')
    print('⚠️  NOW: Runtime → Restart runtime → Then Run All again')
else:
    print('\n❌ SOME packages failed. STOP here and send error.')

In [ ]:
# ==========================================
# CELL 2: ANTI-IDLE
# ==========================================
from IPython.display import display, Javascript

display(Javascript('''
    setInterval(function(){
        var btn = document.querySelector('colab-run-button');
        if(btn) btn.click();
    }, 300000);
'''))
print('✅ Anti-idle active! Safe to close tab after all cells run.')

In [ ]:
# ==========================================
# CELL 3: LOAD QWEN3-30B-A3B + START OPENAI-COMPATIBLE SERVER
# ==========================================
import torch
import time
import threading
import requests
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse
import uvicorn
import nest_asyncio
nest_asyncio.apply()

print('🚀 Loading Qwen3-30B-A3B with 4-bit quantization... (~5-8 min)')

# 4-bit quantization config (fits in T4 16GB easily)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

MODEL_ID = 'Qwen/Qwen3-30B-A3B'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print('✅ Tokenizer loaded')

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)
print('✅ Model loaded in 4-bit')

# Check memory usage
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1024**3
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'📊 VRAM: {mem_used:.1f}GB / {mem_total:.1f}GB ({100*mem_used/mem_total:.1f}%)')

# ==========================================
# OpenAI-compatible FastAPI server
# ==========================================
app = FastAPI(title='OMNIS Qwen3 Server')

@app.get('/health')
async def health():
    return {'status': 'ok'}

@app.get('/v1/models')
async def list_models():
    return {
        'data': [{
            'id': 'qwen3-30b',
            'object': 'model',
            'created': int(time.time()),
            'owned_by': 'local'
        }]
    }

@app.post('/v1/chat/completions')
async def chat_completions(request: dict):
    try:
        messages = request.get('messages', [])
        max_tokens = request.get('max_tokens', 1024)
        temperature = request.get('temperature', 0.7)
        top_p = request.get('top_p', 0.9)
        stream = request.get('stream', False)
        
        # Apply chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer(text, return_tensors='pt').to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode only new tokens
        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        return {
            'id': f'chatcmpl-{int(time.time())}',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': 'qwen3-30b',
            'choices': [{
                'index': 0,
                'message': {
                    'role': 'assistant',
                    'content': response_text
                },
                'finish_reason': 'stop'
            }],
            'usage': {
                'prompt_tokens': int(inputs['input_ids'].shape[1]),
                'completion_tokens': int(len(new_tokens)),
                'total_tokens': int(inputs['input_ids'].shape[1] + len(new_tokens))
            }
        }
    except Exception as e:
        return JSONResponse(status_code=500, content={'error': str(e)})

# Start server in background
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to be ready
for i in range(30):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'✅ LLM Server READY on port 8000 ({(i+1)}s)')
            break
    except:
        pass
    time.sleep(1)
else:
    print('❌ LLM Server failed to start')

In [ ]:
# ==========================================
# CELL 4: START JINA READER SERVER
# ==========================================
import threading, time, requests as req
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn

jina_app = FastAPI(title='OMNIS Jina Reader')

@jina_app.get('/health')
async def jina_health():
    return {'status':'ok'}

@jina_app.get('/extract')
async def extract(url: str = Query(...)):
    try:
        dl = trafilatura.fetch_url(url)
        if not dl:
            return JSONResponse(400, content={'error':'fetch failed','url':url})
        txt = trafilatura.extract(dl, include_comments=False, include_tables=True, no_fallback=False)
        if not txt or len(txt.strip()) < 50:
            return JSONResponse(400, content={'error':'content too short','url':url})
        return {'url':url,'content':txt,'word_count':len(txt.split()),'status':'success'}
    except Exception as e:
        return JSONResponse(500, content={'error':str(e),'url':url})

def run_jina():
    uvicorn.run(jina_app, host='0.0.0.0', port=8001, log_level='warning')

jina_thread = threading.Thread(target=run_jina, daemon=True)
jina_thread.start()
time.sleep(3)

try:
    r = req.get('http://localhost:8001/health', timeout=5)
    print('✅ Jina Reader READY on port 8001' if r.status_code==200 else '❌ Jina error')
except Exception as e:
    print(f'❌ Jina failed: {e}')

In [ ]:
# ==========================================
# CELL 5: CLOUDFLARE TUNNELS
# ==========================================
import subprocess, re

def tunnel(port):
    p = subprocess.Popen(
        ['cloudflared','tunnel','--url',f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return p, m.group(0)
    return p, None

print('🌐 Tunnel LLM (8000)...')
p1, u1 = tunnel(8000)
print('🌐 Tunnel Jina (8001)...')
p2, u2 = tunnel(8001)

if u1 and u2:
    print('\n' + '='*60)
    print('🎉 OMNIS-COURT COLAB READY!')
    print('='*60)
    print(f'🧠 LLM:  {u1}')
    print(f'📖 JINA: {u2}')
    print('='*60)
    print('📋 COPY BOTH URLs → config/platforms.json')
    print('🔒 Anti-idle ON → safe to close tab')
    print('🧪 Test URLs from YOUR browser (not from Colab)')
    print('='*60)
else:
    print(f'❌ Tunnel failed: LLM={u1}, Jina={u2}')

In [ ]:
# ==========================================
# CELL 6: LOCALHOST TESTS
# ==========================================
import requests

print('🧪 Testing LLM on localhost:8000...')
try:
    r = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={'model':'qwen3-30b','messages':[{'role':'user','content':'Say OK if you are Qwen3-30B-A3B'}],'max_tokens':20,'temperature':0.7},
        timeout=120
    )
    if r.status_code == 200:
        resp = r.json()['choices'][0]['message']['content']
        print(f'✅ LLM localhost OK: {resp[:100]}')
    else:
        print(f'❌ LLM localhost: {r.status_code} - {r.text[:200]}')
except Exception as e:
    print(f'❌ LLM localhost: {e}')

print('\n🧪 Testing Jina on localhost:8001...')
try:
    r = requests.get(
        'http://localhost:8001/extract',
        params={'url':'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ Jina localhost OK: {r.json()['word_count']} words")
    else:
        print(f'❌ Jina localhost: {r.status_code}')
except Exception as e:
    print(f'❌ Jina localhost: {e}')

print('\n' + '='*60)
print('🌐 NOW TEST TUNNEL URLs FROM YOUR BROWSER:')
print(f'   {u1}/v1/models')
print(f'   {u2}/health')
print('='*60)